<a href="https://colab.research.google.com/github/ACM-BRIN/peatland-fire-prediction/blob/main/Code_Jambu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Library

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from datetime import datetime, timedelta

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

import lightgbm as lgbm

from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from keras import Model, Sequential
from keras.layers import Dense, Dropout
from keras.losses import MeanSquaredError

!pip3 install ann_visualizer
from ann_visualizer.visualize import ann_viz

!pip install shap
import shap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Helper Function

In [ ]:
pair = [['pcp','rainfall','mm'],  ['t2m','near surface temperature','deg C'],
        ['rh','relative humidity','%'], ['ws','wind speed','m/s'],
        ['s_temp','soil temperature','deg C'], ['s_moist','soil moisture','%'], ['gwl','ground water level','cm'],
        ['hotspot','hotspot','spot'], ['emission','emission','g C/m^2']]

col_unit = pd.DataFrame(pair, columns=['Column','Aliases','Unit']).set_index('Column')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 30
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.direction'],plt.rcParams['xtick.major.size'],plt.rcParams['xtick.major.width']='in',15,5
plt.rcParams['ytick.direction'],plt.rcParams['ytick.major.size'],plt.rcParams['ytick.major.width']='in',15,5
plt.rcParams['lines.linewidth']=3

def wmi(atribut, column, case): #--> Window Mean Interpolation
    if case == 1 : # --> (Nan/(will be replace), int1, int2, int3, int4, int5, int6)
      convert = atribut.loc[t + timedelta(days=1) : t + timedelta(days=6), column].mean()

    elif case == 2 : # --> (int1, int2, int3, Nan/(will be replace), int4, int5, int6)
      dtindex = pd.date_range(start= t - timedelta(days=3), end = t + timedelta(days=3), freq='d').delete(3)
      convert = atribut.loc[dtindex,column].mean()

    elif case == 3 : # --> (int1, int2, int3, int4, int5, int6,  Nan/(will be replace))
      convert = atribut.loc[t - timedelta(days=6) : t - timedelta(days=1),column].mean()

    return convert

def scale_predictor(X_train, X_test):
    scaler_x = MinMaxScaler()
    X_train_scaled = scaler_x.fit_transform(X_train)
    X_test_scaled = scaler_x.transform(X_test)
    return X_train_scaled, X_test_scaled

# Creating model using the Sequential in tensorflow
def build_nn(X_train):
    model = Sequential()

    model.add(Dense(X_train.shape[1]+4, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(X_train.shape[1]+4, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(X_train.shape[1]+4, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(y_train.shape[1], activation='linear'))

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics = ['mse'])
    return model

def fit_nn(model, X_train):
    early_stop = keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 10)
    history = model.fit(X_train, y_train, epochs = 100,
                        validation_split = 0.2, batch_size = 128,
                        shuffle = False, callbacks = [early_stop])
    return history

def plot_history(history):
    fig = plt.figure(figsize=(15, 6))
    plt.plot(history.history['loss'], label='Train loss')
    plt.plot(history.history['val_loss'], label='validation loss')
    plt.legend(loc='upper right')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')

def convert(y_pred):
    for i in range (len(y_pred)):
        if y_pred[i,0] < 0 :
            y_pred[i,0]  = 0
        else :
            y_pred[i,0] = round(y_pred[i,0])

        if y_pred[i,1] < 0 :
            y_pred[i,1]  = 0
    return(y_pred)

def plot_experiment(y_test, y_pred_1, y_pred_2, y_pred_3):
    fig, ax = plt.subplots(2, figsize=(16,16))
    ax[0].plot(y_test[:,0], linewidth=6, label='Actual')
    ax[0].plot(y_pred_1[:,0], '-o', label='Meteorological')
    ax[0].plot(y_pred_2[:,0], '-x', label='Hydrological')
    ax[0].plot(y_pred_3[:,0], '-+', label='All')
    ax[0].set(ylabel='Hotspot')
    ax[0].legend(fontsize = 20, loc = 'upper right')
    ax[0].set_title('(a)')

    ax[1].plot(y_test[:,1], linewidth=6, label='Actual')
    ax[1].plot(y_pred_1[:,1], '-o', label='Meteorological')
    ax[1].plot(y_pred_2[:,1], '-x', label='Hydrological')
    ax[1].plot(y_pred_3[:,1], '-+', label='All')
    ax[1].set(xlabel='Sample', ylabel='Emission (g C/m^2)')
    ax[1].set_title('(b)')

def evaluate(y_test, y_pred):
    mae_hotspot = round(mean_absolute_error(y_test[:,0], y_pred[:,0]), 2)
    nrmse_hotspot = round(mean_squared_error(y_test[:,0], y_pred[:,0], squared=False)/(max(y_test[:,0])-min(y_test[:,0])), 2)
    cc_hotspot = round(np.corrcoef(y_test[:,0], y_pred[:,0])[0][1], 2)

    mae_emission = round(mean_absolute_error(y_test[:,1], y_pred[:,1]), 2)
    nrmse_emission = round(mean_squared_error(y_test[:,1], y_pred[:,1], squared=False)/(max(y_test[:,1])-min(y_test[:,1])), 2)
    cc_emission = round(np.corrcoef(y_test[:,1], y_pred[:,1])[0][1], 2)

    metrics = {'':['Hotspot','Emission'],
               'MAE': [mae_hotspot, mae_emission],
               'NRMSE':[nrmse_hotspot, nrmse_emission],
               'CC':[cc_hotspot, cc_emission]}

    metrics_table = pd.DataFrame(metrics).set_index('')
    return(metrics_table)

# Step 1 : Read and Explore Data

In [ ]:
# Read File
raw_gabungan = pd.read_excel('./Data/Data gabungan.xlsx',
                         sheet_name = 'Jambu', parse_dates = ['Date'], index_col = 'Date')
raw_hotspot = pd.read_excel('./Data/Data_hotspot.xlsx',
                         sheet_name = 'Jambu', parse_dates = ['Date'], index_col = 'Date')
start = '2019-08-22'
end = '2020-12-31'
raw = raw_gabungan[start:end].join(raw_hotspot)
raw['Hotspot'] = raw['Hotspot'].fillna(0)

raw['ws'] = (raw['uw']**2 + raw['vw']**2)**(1/2)

df = raw[['curah_hujan','t2m','rh','ws','soil_moisture','tma','Hotspot','daily_emission']]
df.columns = ['pcp','t2m','rh','ws','s_moist','gwl','hotspot','emission']

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(36,24))

for i, ax in enumerate(axs.flat):
    ax.plot(df.iloc[:,i], color ='blue', label=col_unit.loc[df.columns[i], 'Aliases'])
    ax.set_xticks(pd.date_range((datetime.strptime(start, '%Y-%m-%d')+ pd.DateOffset(months=1)).strftime('%Y-%m-%d'),end , freq='3M')-pd.offsets.MonthBegin(1))
    ax.set_xticklabels((pd.date_range((datetime.strptime(start, '%Y-%m-%d')+ pd.DateOffset(months=1)).strftime('%Y-%m-%d'),end , freq='3M')-pd.offsets.MonthBegin(1)).strftime('%Y-%m'),
                       fontsize=25)

    ax.set_xlabel('Date')
    ax.set_ylabel(col_unit.loc[df.columns[i], 'Unit'])
    ax.legend(loc = 'upper right')

In [ ]:
# Descriptive statistics for each column
df.describe().round(4)

In [ ]:
df.corr(method='pearson')

# Step 2 : Data Pre-Processing

## Handling Error and Missing Values

Interpolate Error and Missing Values using Window Mean Interpolation

In [ ]:
for t in pd.date_range(start='2019-08-22', end ='2019-08-22', freq='d'):
    df.loc[t]['s_moist'] = round(wmi(df, 's_moist', 1), 3)

for t in pd.date_range(start='2020-04-21', end ='2020-09-15', freq='d'):
    df.loc[t]['s_moist'] = round(wmi(df, 's_moist', 3), 3)

for t in pd.date_range(start='2020-09-17', end ='2020-12-31', freq='d'):
    df.loc[t]['s_moist'] = round(wmi(df, 's_moist', 3), 3)

for t in pd.date_range(start='2020-10-14', end ='2020-12-31', freq='d'):
    df.loc[t]['gwl'] = round(wmi(df, 'gwl', 3), 3)

for t in pd.date_range(start='2020-12-30', end ='2020-12-31', freq='d'):
    df.loc[t]['pcp'] = round(wmi(df, 'pcp', 3), 3)

In [ ]:
# Check missing values
df.isnull().sum()

## Final Dataset

The Dataset after interpolate

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(36,24))

for i, ax in enumerate(axs.flat):
    ax.plot(df.iloc[:,i], color ='blue', label=col_unit.loc[df.columns[i], 'Aliases'])
    ax.set_xticks(pd.date_range((datetime.strptime(start, '%Y-%m-%d')+ pd.DateOffset(months=1)).strftime('%Y-%m-%d'),end , freq='3M')-pd.offsets.MonthBegin(1))
    ax.set_xticklabels((pd.date_range((datetime.strptime(start, '%Y-%m-%d')+ pd.DateOffset(months=1)).strftime('%Y-%m-%d'),end , freq='3M')-pd.offsets.MonthBegin(1)).strftime('%Y-%m'),
                       fontsize=25)

    ax.set_xlabel('Date')
    ax.set_ylabel(col_unit.loc[df.columns[i], 'Unit'])
    ax.legend(loc = 'upper right')

In [ ]:
#Creating subplot of each column with its own scale
red_circle = dict(markerfacecolor='red', marker='o', markeredgecolor='white')

fig, axs = plt.subplots(len(df.columns)-2, 1, figsize=(16,14))
for i, ax in enumerate(axs.flat):
    ax.boxplot(df.iloc[:,i], widths=0.3, vert=False, notch=True, patch_artist=True,
               boxprops=dict(facecolor='red', alpha=0.5), capprops=dict(color='red', linewidth=2), whiskerprops=dict(color='red', linewidth=2),
               flierprops=dict(markerfacecolor='red', marker='o', markeredgecolor='white', markersize= 15),
               medianprops=dict(color='red', linewidth=2)
            )
    ax.set_yticklabels([df.columns[i]])
plt.tight_layout()

In [ ]:
fig, ax1 = plt.subplots(figsize=(16,8))
ax2 = ax1.twinx()

start='2019-08-22'
end='2020-01-31'

ax1.set_xticks(pd.date_range((datetime.strptime(start, '%Y-%m-%d')+ pd.DateOffset(months=1)).strftime('%Y-%m-%d'),end , freq='1M')-pd.offsets.MonthBegin(1))
ax1.set_xticklabels((pd.date_range((datetime.strptime(start, '%Y-%m-%d')+ pd.DateOffset(months=1)).strftime('%Y-%m-%d'),end , freq='1M')-pd.offsets.MonthBegin(1)).strftime('%Y-%m'),
                       fontsize=25)
ax2.set_xticks(pd.date_range((datetime.strptime(start, '%Y-%m-%d')+ pd.DateOffset(months=1)).strftime('%Y-%m-%d'),end , freq='1M')-pd.offsets.MonthBegin(1))
ax2.set_xticklabels((pd.date_range((datetime.strptime(start, '%Y-%m-%d')+ pd.DateOffset(months=1)).strftime('%Y-%m-%d'),end , freq='1M')-pd.offsets.MonthBegin(1)).strftime('%Y-%m'),
                       fontsize=25)

ax2.spines['left'].set_color('blue')
ax2.spines['right'].set_color('red')

ax1.tick_params(axis='y', colors='blue')
ax2.tick_params(axis='y', colors='red')

lns1=ax1.plot(df['hotspot'], '-b', label='Hotspot', linewidth=6)
lns2=ax2.plot(df['emission'], '-r', label='Emission')

lns = lns1+lns2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc=0)

ax1.set_xlabel('Date')
ax1.set_ylabel('Spot')
ax2.set_ylabel('g C/m^2')

ax1.set_xlim(datetime.strptime(start, '%Y-%m-%d'),datetime.strptime(end, '%Y-%m-%d'))

In [ ]:
# Descriptive statistics for each column
df.describe().round(4)

In [ ]:
df.corr(method='pearson')

## Prepare Predictor and Target

Determine the variables that will be used as predictors in the experimental scheme

In [ ]:
pred_meteorological = ['pcp','t2m','rh','ws']
pred_hydrological = ['s_moist','gwl']
pred_all = pred_meteorological + pred_hydrological
target = ['hotspot','emission']

In [ ]:
X_1 = df[pred_meteorological]
X_2 = df[pred_hydrological]
X_3 = df[pred_all]
y = df[target]

print(X_1.shape)
print(X_2.shape)
print(X_3.shape)
print(y.shape)

In [ ]:
X_train_1, X_test_1, y_train, y_test = train_test_split(X_1, y, test_size = 0.2, random_state=42)
X_train_2, X_test_2, y_train, y_test = train_test_split(X_2, y, test_size = 0.2, random_state=42)
X_train_3, X_test_3, y_train, y_test = train_test_split(X_3, y, test_size = 0.2, random_state=42)

In [ ]:
X_train_1, X_test_1 = scale_predictor(X_train_1, X_test_1)
X_train_2, X_test_2 = scale_predictor(X_train_2, X_test_2)
X_train_3, X_test_3 = scale_predictor(X_train_3, X_test_3)

y_train = np.array(y_train)
y_test = np.array(y_test)

print('X_train_1.shape:', X_train_1.shape, ' | X_test_1.shape:', X_test_1.shape)
print('X_train_2.shape:', X_train_2.shape, ' | X_test_2.shape:', X_test_2.shape)
print('X_train_3.shape:', X_train_3.shape, ' | X_test_3.shape:', X_test_3.shape)
print('y_train.shape: ', y_train.shape, ' | y_test.shape: ', y_test.shape)

# Step 3 : Experiment

Find the best predictor for each Method (Random Forest, Light Gradient Boosted Machine, and Neural Network)

## RF

In [ ]:
from pprint import pprint

from sklearn.model_selection import RandomizedSearchCV
# Method of selecting samples for training each tree
bootstrap = [True, False]
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 100, num = 10)]
max_depth.append(None)
# Number of features to consider at every split
max_features = ['auto', 'sqrt']
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]

# Create the random grid
random_grid = {'estimator__bootstrap': bootstrap,
               'estimator__max_depth': max_depth,
               'estimator__max_features': max_features,
               'estimator__min_samples_leaf': min_samples_leaf,
               'estimator__min_samples_split': min_samples_split,
               'estimator__n_estimators': n_estimators
               }
pprint(random_grid)

### Exp 1

In [ ]:
rf_base_1 = MultiOutputRegressor(RandomForestRegressor(random_state = 42))
rf_base_1.fit(X_train_1, y_train)
pred_rf_base_1 = convert(rf_base_1.predict(X_test_1))

base_accuracy_1 = evaluate(y_test, pred_rf_base_1)['MAE'].mean()
evaluate(y_test, pred_rf_base_1)

In [ ]:
from pprint import pprint
# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(rf_base_1.get_params())

In [ ]:
# Use the random grid to search for best hyperparameters
# Random search of parameters, using 3 fold cross validation,
# search across 100 different combinations, and use all available cores
rf_random_1 = RandomizedSearchCV(estimator = MultiOutputRegressor(RandomForestRegressor()), param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)

In [ ]:
# Fit the random search model
rf_random_1.fit(X_train_1, y_train)

rf_random_1.best_params_

In [ ]:
best_random_1 = rf_random_1.best_estimator_
pred_rf_random_1 = convert(best_random_1.predict(X_test_1))

random_accuracy_1 = evaluate(y_test, pred_rf_random_1)['MAE'].mean()
evaluate(y_test, pred_rf_random_1)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_1 - random_accuracy_1) / base_accuracy_1))

In [ ]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search
param_grid = {
    'estimator__bootstrap': [True],
    'estimator__max_depth': [100],
    'estimator__max_features': ['sqrt'],
    'estimator__min_samples_leaf': [3, 4],
    'estimator__min_samples_split': [8, 10],
    'estimator__n_estimators': [1800]
}

# Instantiate the grid search model
grid_search_1 = GridSearchCV(estimator = MultiOutputRegressor(RandomForestRegressor()), param_grid = param_grid,
                          cv = 3, n_jobs = -1, verbose = 2)

In [ ]:
# Fit the grid search to the data
grid_search_1.fit(X_train_1, y_train)

grid_search_1.best_params_

In [ ]:
model_rf_1 = grid_search_1.best_estimator_
pred_rf_1 = convert(model_rf_1.predict(X_test_1))

grid_accuracy_1 = evaluate(y_test, pred_rf_1)['MAE'].mean()
evaluate(y_test, pred_rf_1)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_1 - grid_accuracy_1) / base_accuracy_1))

### Exp 2

In [ ]:
rf_base_2 = MultiOutputRegressor(RandomForestRegressor(random_state = 42))
rf_base_2.fit(X_train_2, y_train)
pred_rf_base_2 = convert(rf_base_2.predict(X_test_2))

base_accuracy_2 = evaluate(y_test, pred_rf_base_2)['MAE'].mean()
evaluate(y_test, pred_rf_base_2)

In [ ]:
from pprint import pprint
# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(rf_base_2.get_params())

In [ ]:
# Use the random grid to search for best hyperparameters
# Random search of parameters, using 3 fold cross validation,
# search across 100 different combinations, and use all available cores
rf_random_2 = RandomizedSearchCV(estimator = MultiOutputRegressor(RandomForestRegressor()), param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)

In [ ]:
# Fit the random search model
rf_random_2.fit(X_train_2, y_train)

rf_random_2.best_params_

In [ ]:
best_random_2 = rf_random_2.best_estimator_
pred_rf_random_2 = convert(best_random_2.predict(X_test_2))

random_accuracy_2 = evaluate(y_test, pred_rf_random_2)['MAE'].mean()
evaluate(y_test, pred_rf_random_2)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_2 - random_accuracy_2) / base_accuracy_2))

In [ ]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search
param_grid = {
    'estimator__bootstrap': [True],
    'estimator__max_depth': [80, 90, 100],
    'estimator__max_features': ['sqrt'],
    'estimator__min_samples_leaf': [3, 4, 5],
    'estimator__min_samples_split': [8, 16, 32],
    'estimator__n_estimators': [1200, 1400, 1600]
}

# Instantiate the grid search model
grid_search_2 = GridSearchCV(estimator = MultiOutputRegressor(RandomForestRegressor()), param_grid = param_grid,
                          cv = 3, n_jobs = -1, verbose = 2)

In [ ]:
# Fit the grid search to the data
grid_search_2.fit(X_train_2, y_train)

grid_search_2.best_params_

In [ ]:
model_rf_2 = grid_search_2.best_estimator_
pred_rf_2 = convert(model_rf_2.predict(X_test_2))

grid_accuracy_2 = evaluate(y_test, pred_rf_2)['MAE'].mean()
evaluate(y_test, pred_rf_2)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_2 - grid_accuracy_2) / base_accuracy_2))

### Exp 3

In [ ]:
rf_base_3 = MultiOutputRegressor(RandomForestRegressor(random_state = 42))
rf_base_3.fit(X_train_3, y_train)
pred_rf_base_3 = convert(rf_base_3.predict(X_test_3))

base_accuracy_3 = evaluate(y_test, pred_rf_base_3)['MAE'].mean()
evaluate(y_test, pred_rf_base_3)

In [ ]:
from pprint import pprint
# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(rf_base_3.get_params())

In [ ]:
# Use the random grid to search for best hyperparameters
# Random search of parameters, using 3 fold cross validation,
# search across 100 different combinations, and use all available cores
rf_random_3 = RandomizedSearchCV(estimator = MultiOutputRegressor(RandomForestRegressor()), param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)

In [ ]:
# Fit the random search model
rf_random_3.fit(X_train_3, y_train)

rf_random_3.best_params_

In [ ]:
best_random_3 = rf_random_3.best_estimator_
pred_rf_random_3 = convert(best_random_3.predict(X_test_3))

random_accuracy_3 = evaluate(y_test, pred_rf_random_3)['MAE'].mean()
evaluate(y_test, pred_rf_random_3)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_3 - random_accuracy_3) / base_accuracy_3))

In [ ]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search
param_grid = {
    'estimator__bootstrap': [True],
    'estimator__max_depth': [80, 90, 100],
    'estimator__max_features': ['sqrt'],
    'estimator__min_samples_leaf': [3, 4],
    'estimator__min_samples_split': [8, 10, 12],
    'estimator__n_estimators': [1600, 1800, 2000]
}

# Instantiate the grid search model
grid_search_3 = GridSearchCV(estimator = MultiOutputRegressor(RandomForestRegressor()), param_grid = param_grid,
                          cv = 3, n_jobs = -1, verbose = 2)

In [ ]:
# Fit the grid search to the data
grid_search_3.fit(X_train_3, y_train)

grid_search_3.best_params_

In [ ]:
model_rf_3 = grid_search_3.best_estimator_
pred_rf_3 = convert(model_rf_3.predict(X_test_3))

grid_accuracy_3 = evaluate(y_test, pred_rf_3)['MAE'].mean()
evaluate(y_test, pred_rf_3)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_3 - grid_accuracy_3) / base_accuracy_3))

### Plot

In [ ]:
plot_experiment(y_test, pred_rf_1, pred_rf_2, pred_rf_3)

In [ ]:
evaluate(y_test, pred_rf_1)

In [ ]:
evaluate(y_test, pred_rf_2)

In [ ]:
evaluate(y_test, pred_rf_3)

## LightGBM

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# Boosting Type
boosting_type = ['gbdt', 'dart', 'goss']
# maximum depth of the tree
max_depth = [5,6,7,8,9,10,12]
# maximum number of leaves in one tree, main parameter to tune for a tree model
num_leaves = [int(x) for x in np.linspace(start = 20, stop = 300, num = 10)]
# Number of trees in lgbm
n_estimators = [int(x) for x in np.linspace(start = 100, stop = 1000, num = 10)]
# shrinkage rate, determine how fast the model can learn
learning_rate = [0.1, 0.01, 0.001]

# Create the random grid
random_grid = {'estimator__boosting_type': boosting_type,
               'estimator__max_depth': max_depth,
               'estimator__num_leaves': num_leaves,
               'estimator__n_estimators':n_estimators,
               'estimator__learning_rate': learning_rate
                }
pprint(random_grid)

### Exp 1

In [ ]:
lgbm_base_1 = MultiOutputRegressor(lgbm.LGBMRegressor())
lgbm_base_1.fit(X_train_1, y_train)
pred_lgbm_base_1 = convert(lgbm_base_1.predict(X_test_1))

base_accuracy_1 = evaluate(y_test, pred_lgbm_base_1)['MAE'].mean()
evaluate(y_test, pred_lgbm_base_1)

In [ ]:
from pprint import pprint
# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(lgbm_base_1.get_params())

In [ ]:
# Use the random grid to search for best hyperparameters
# Random search of parameters, using 3 fold cross validation,
# search across 100 different combinations, and use all available cores
lgbm_random_1 = RandomizedSearchCV(estimator = MultiOutputRegressor(lgbm.LGBMRegressor()), param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)

In [ ]:
# Fit the random search model
lgbm_random_1.fit(X_train_1, y_train)

lgbm_random_1.best_params_

In [ ]:
best_random_1 = lgbm_random_1.best_estimator_
pred_lgbm_random_1 = convert(best_random_1.predict(X_test_1))

random_accuracy_1 = evaluate(y_test, pred_lgbm_random_1)['MAE'].mean()
evaluate(y_test, pred_lgbm_random_1)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_1 - random_accuracy_1) / base_accuracy_1))

In [ ]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search
param_grid = {'estimator__boosting_type': ['dart'],
              'estimator__max_depth': [11, 12, 13],
              'estimator__num_leaves': [200, 250, 300],
              'estimator__n_estimators': [100, 200, 300],
              'estimator__learning_rate': [0.01]
}
# Create a based model
# rf = RandomForestRegressor()
# Instantiate the grid search model
grid_search_1 = GridSearchCV(estimator = MultiOutputRegressor(lgbm.LGBMRegressor()), param_grid = param_grid,
                          cv = 3, n_jobs = -1, verbose = 2)

In [ ]:
# Fit the grid search to the data
grid_search_1.fit(X_train_1, y_train)

grid_search_1.best_params_

In [ ]:
model_lgbm_1 = grid_search_1.best_estimator_
pred_lgbm_1 = convert(model_lgbm_1.predict(X_test_1))

grid_accuracy = evaluate(y_test, pred_lgbm_1)['MAE'].mean()
evaluate(y_test, pred_lgbm_1)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_1 - grid_accuracy_1) / base_accuracy_1))

### Exp 2

In [ ]:
lgbm_base_2 = MultiOutputRegressor(lgbm.LGBMRegressor())
lgbm_base_2.fit(X_train_2, y_train)
pred_lgbm_base_2 = convert(lgbm_base_2.predict(X_test_2))

base_accuracy_2 = evaluate(y_test, pred_lgbm_base_2)['MAE'].mean()
evaluate(y_test, pred_lgbm_base_2)

In [ ]:
from pprint import pprint
# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(lgbm_base_2.get_params())

In [ ]:
# Use the random grid to search for best hyperparameters
# Random search of parameters, using 3 fold cross validation,
# search across 100 different combinations, and use all available cores
lgbm_random_2 = RandomizedSearchCV(estimator = MultiOutputRegressor(lgbm.LGBMRegressor()), param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)

In [ ]:
# Fit the random search model
lgbm_random_2.fit(X_train_2, y_train)

lgbm_random_2.best_params_

In [ ]:
best_random_2 = lgbm_random_2.best_estimator_
pred_lgbm_random_2 = convert(best_random_2.predict(X_test_2))

random_accuracy_2 = evaluate(y_test, pred_lgbm_random_2)['MAE'].mean()
evaluate(y_test, pred_lgbm_random_2)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_2 - random_accuracy_2) / base_accuracy_2))

In [ ]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search
param_grid = {'estimator__boosting_type': ['dart'],
              'estimator__max_depth': [8, 9, 10],
              'estimator__num_leaves': [100, 150, 200],
              'estimator__n_estimators': [100, 200, 300],
              'estimator__learning_rate': [0.01]
}
# Create a based model
# rf = RandomForestRegressor()
# Instantiate the grid search model
grid_search_2 = GridSearchCV(estimator = MultiOutputRegressor(lgbm.LGBMRegressor()), param_grid = param_grid,
                          cv = 3, n_jobs = -1, verbose = 2)

In [ ]:
# Fit the grid search to the data
grid_search_2.fit(X_train_2, y_train)

grid_search_2.best_params_

In [ ]:
model_lgbm_2 = grid_search_2.best_estimator_
pred_lgbm_2 = convert(model_lgbm_2.predict(X_test_2))

grid_accuracy_2 = evaluate(y_test, pred_lgbm_2)['MAE'].mean()
evaluate(y_test, pred_lgbm_2)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_2 - grid_accuracy_2) / base_accuracy_2))

### Exp 3

In [ ]:
lgbm_base_3 = MultiOutputRegressor(lgbm.LGBMRegressor())
lgbm_base_3.fit(X_train_3, y_train)
pred_lgbm_base_3 = convert(lgbm_base_3.predict(X_test_3))

base_accuracy_3 = evaluate(y_test, pred_lgbm_base_3)['MAE'].mean()
evaluate(y_test, pred_lgbm_base_3)

In [ ]:
from pprint import pprint
# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(lgbm_base_3.get_params())

In [ ]:
# Use the random grid to search for best hyperparameters
# Random search of parameters, using 3 fold cross validation,
# search across 100 different combinations, and use all available cores
lgbm_random_3 = RandomizedSearchCV(estimator = MultiOutputRegressor(lgbm.LGBMRegressor()), param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)

In [ ]:
# Fit the random search model
lgbm_random_3.fit(X_train_3, y_train)

lgbm_random_3.best_params_

In [ ]:
best_random_3 = lgbm_random_3.best_estimator_
pred_lgbm_random_3 = convert(best_random_3.predict(X_test_3))

random_accuracy_3 = evaluate(y_test, pred_lgbm_random_3)['MAE'].mean()
evaluate(y_test, pred_lgbm_random_3)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_3 - random_accuracy_3) / base_accuracy_3))

In [ ]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search
param_grid = {'estimator__boosting_type': ['dart'],
              'estimator__max_depth': [11, 12, 13],
              'estimator__num_leaves': [200, 250, 300],
              'estimator__n_estimators': [100, 200, 300],
              'estimator__learning_rate': [0.01]
}
# Create a based model
# rf = RandomForestRegressor()
# Instantiate the grid search model
grid_search_3 = GridSearchCV(estimator = MultiOutputRegressor(lgbm.LGBMRegressor()), param_grid = param_grid,
                          cv = 3, n_jobs = -1, verbose = 2)

In [ ]:
# Fit the grid search to the data
grid_search_3.fit(X_train_3, y_train)

grid_search_3.best_params_

In [ ]:
model_lgbm_3 = grid_search_3.best_estimator_
pred_lgbm_3 = convert(model_lgbm_3.predict(X_test_3))

grid_accuracy_3 = evaluate(y_test, pred_lgbm_3)['MAE'].mean()
evaluate(y_test, pred_lgbm_3)

In [ ]:
print('Improvement : MAE.mean() decreased by {:0.2f}%'.format( 100 * (base_accuracy_3 - grid_accuracy_3) / base_accuracy_3))

### Plot

In [ ]:
plot_experiment(y_test, pred_lgbm_1, pred_lgbm_2, pred_lgbm_3)

In [ ]:
evaluate(y_test, pred_lgbm_1)

In [ ]:
evaluate(y_test, pred_lgbm_2)

In [ ]:
evaluate(y_test, pred_lgbm_3)

## NN

### Exp 1

In [ ]:
# Model NN
model_nn_1 = build_nn(X_train_1)
history_1 = fit_nn(model_nn_1, X_train_1)
plot_history(history_1)

In [ ]:
pred_nn_1 = convert(model_nn_1.predict(X_test_1))

### Exp 2

In [ ]:
# Model NN
model_nn_2 = build_nn(X_train_2)
history_2 = fit_nn(model_nn_2, X_train_2)
plot_history(history_2)

In [ ]:
pred_nn_2 = convert(model_nn_2.predict(X_test_2))

### Exp 3

In [ ]:
# Model NN
model_nn_3 = build_nn(X_train_3)

history_3 = fit_nn(model_nn_3, X_train_3)
plot_history(history_3)

In [ ]:
pred_nn_3 = convert(model_nn_3.predict(X_test_3))

### Plot

In [ ]:
plot_experiment(y_test, pred_nn_1, pred_nn_2, pred_nn_3)

In [ ]:
evaluate(y_test, pred_nn_1)

In [ ]:
evaluate(y_test, pred_nn_2)

In [ ]:
evaluate(y_test, pred_nn_3)

## Best Exp All Method
RF --> Exp 3

LightGBM --> Exp 2

NN --> Exp 3

In [ ]:
model_rf = model_rf_3
pred_rf = convert(model_rf.predict(X_test_3))

model_lgbm = model_lgbm_2
pred_lgbm = convert(model_lgbm.predict(X_test_2))

model_nn = model_nn_3
pred_nn = convert(model_nn.predict(X_test_3))

In [ ]:
ann_viz(model_nn, view=True, title='model_nn', filename='model_nn')

In [ ]:
plt.figure(figsize=(16, 8))
plt.plot(y_test[:,0], linewidth=6, label='Actual')
plt.plot(pred_rf[:,0], '-o', label='RF')
plt.plot(pred_lgbm[:,0], '-x', label='LightGBM')
plt.plot(pred_nn[:,0], '-+', label='NN')
plt.xlabel('Sample')
plt.ylabel('Spot')
plt.legend(fontsize = 20, loc = 'upper right')

plt.figure(figsize=(16, 8))
plt.plot(y_test[:,1], linewidth=6, label='Actual')
plt.plot(pred_rf[:,1], '-o', label='RF')
plt.plot(pred_lgbm[:,1], '-x', label='LightGBM')
plt.plot(pred_nn[:,1], '-+', label='NN')
plt.xlabel('Sample')
plt.ylabel('g C/m^2')
plt.legend(fontsize = 20, loc = 'upper right')

# Feature Importance

Find the important variables by using RandomForest.feature_importances_

## RF

In [ ]:
# Get numerical feature importances
rf_importances_0 = model_rf_3.estimators_[0].feature_importances_
rf_importances_1 = model_rf_3.estimators_[1].feature_importances_

print(np.round_(rf_importances_0,3))
print(np.round_(rf_importances_1,3))

## LightGBM

In [ ]:
# Get numerical feature importances
lgbm_importances_0 = model_lgbm_3.estimators_[0].feature_importances_
lgbm_importances_0 = [lgbm_importances_0[i]/sum(lgbm_importances_0) for i in range(len(pred_all))]

lgbm_importances_1 = model_lgbm_3.estimators_[1].feature_importances_
lgbm_importances_1 = [lgbm_importances_1[i]/sum(lgbm_importances_1) for i in range(len(pred_all))]

print(np.round_(lgbm_importances_0,3))
print(np.round_(lgbm_importances_1,3))

## NN

In [ ]:
e = shap.KernelExplainer(model_nn_3, X_train_3)
shap_values = e.shap_values(X_test_3)

In [ ]:
# Get numerical feature importances
nn_importances_0 = abs(shap_values[0]).mean(axis=0)
nn_importances_0 = [nn_importances_0[i]/sum(nn_importances_0) for i in range(len(pred_all))]

nn_importances_1 = abs(shap_values[1]).mean(axis=0)
nn_importances_1 = [nn_importances_1[i]/sum(nn_importances_1) for i in range(len(pred_all))]

print(np.round_(nn_importances_0,3))
print(np.round_(nn_importances_1,3))

## Best Model

In [ ]:
# Get numerical feature importances
best_importances_0 = model_lgbm_2.estimators_[0].feature_importances_
best_importances_0 = [best_importances_0[i]/sum(best_importances_0) for i in range(len(pred_hydrological))]
best_importances_1 = model_rf_3.estimators_[1].feature_importances_

print(np.round_(best_importances_0,3))
print(np.round_(best_importances_1,3))

## Plot

In [ ]:
x = np.arange(len(pred_all))
x_best = np.arange(len(pred_hydrological))

plt.figure(figsize=(16, 8))
plt.bar(x-0.3, rf_importances_0, width=0.2, color='b', align='center')
plt.bar(x-0.1, lgbm_importances_0, width=0.2, color='g', align='center')
plt.bar(x+0.1, nn_importances_0, width=0.2, color='r', align='center')
plt.bar(x_best+4.3, best_importances_0, width=0.2, color='g', hatch='x', align='center')

plt.legend(('RF', 'LightGBM', 'NN', 'Best Model'), fontsize = 20, loc = 'upper left')
plt.xticks(x, pred_all)
plt.ylabel('Importance')

plt.figure(figsize=(16, 8))
plt.bar(x-0.3, rf_importances_1, width=0.2, color='b', align='center')
plt.bar(x-0.1, lgbm_importances_1, width=0.2, color='g', align='center')
plt.bar(x+0.1, nn_importances_1, width=0.2, color='r', align='center')
plt.bar(x+0.3, best_importances_1, width=0.2, color='b', hatch='x', align='center')

plt.legend(('RF', 'LightGBM', 'NN', 'Best Model'), fontsize = 20, loc = 'upper left')
plt.xticks(x, pred_all)
plt.ylabel('Importance')